# Installing libs

In [ ]:
%pip uninstall datasets -y
%pip install datasets==2.14.6
%pip install fsspec==2023.10.0
%pip install pandas pillow requests tqdm pyarrow

# Importing libs

In [ ]:
from datasets import load_dataset
from PIL import Image
from io import BytesIO
from tqdm import tqdm
import os

# Downloading Images

In [ ]:
target_images_count = 50000 # you can change this to any number you want

output_dir = "/content/yfcc100m_50k"

os.makedirs(output_dir, exist_ok=True)

print("Loading dataset stream...")

dataset = load_dataset(
    "dalle-mini/YFCC100M_OpenAI_subset", # official dataset by OpenAI to train dalle-mini
    split="train",
    streaming=True
)

print("Saving images...\n")

saved = 0
failed = 0

for row in tqdm(dataset, total=target_images_count):

    if saved >= target_images_count:
        break

    try:

        img_bytes = row["img"]

        # bytes -> PIL image
        image = Image.open(
            BytesIO(img_bytes)
        ).convert("RGB")

        photo_id = row.get(
            "photoid",
            str(saved)
        )

        save_path = os.path.join(
            output_dir,
            f"{photo_id}.jpg"
        )

        image.save(
            save_path,
            "JPEG",
            quality=95
        )

        saved += 1

    except Exception as e:

        failed += 1

        if failed < 10:
            print("FAILED:", e)

print("Saved :", saved)
print("Failed:", failed)
print("Output:", output_dir)